# Ces scripts servent à ajouter les verbes empruntés au français au lexique final

##  <span style="color: #FFFFE0; font-weight: bold;">Adaptation des lexiques français</span>

In [5]:
import pandas as pd
import os

# Chemin du répertoire de sortie
output_directory = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Lexiques\Lexiques radicaux infinitifs"

# Liste des chemins des fichiers TSV
file_paths = [
    r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Lexiques\Lexiques adaptés\analog.tsv",
    r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Lexiques\Lexiques adaptés\GLAFF.tsv",
    r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Lexiques\Lexiques adaptés\LEFFF.tsv"
]

# Fonction pour traiter chaque fichier
def filter_infinitives(file_path, output_dir):
    df = pd.read_csv(file_path, sep='\t')
    # Filtrer pour ne garder que les verbes en "er" à l'infinitif
    filtered_df = df[(df['mode'] == 'Infinitif') & (df['lemme'].str.endswith('er'))][['lemme']]
    file_name = os.path.basename(file_path)
    output_file_path = os.path.join(output_dir, file_name)
    filtered_df.to_csv(output_file_path, sep='\t', index=False)
    return output_file_path

# Traiter chaque fichier et sauvegarder les résultats dans le répertoire de sortie
filtered_file_paths = [filter_infinitives(file_path, output_directory) for file_path in file_paths]

# Lire les fichiers filtrés et les concaténer
data_frames = [pd.read_csv(file_path, sep='\t') for file_path in filtered_file_paths]
merged_df = pd.concat(data_frames, ignore_index=True)

# Supprimer les doublons
merged_df = merged_df.drop_duplicates()

# Fonction pour extraire le radical des verbes du premier groupe
def extract_radical(lemme):
    if lemme.endswith('er'):
        return lemme[:-2]
    else:
        return lemme

# Appliquer la fonction pour créer la colonne "radical"
merged_df['radical'] = merged_df['lemme'].apply(extract_radical)

# Sauvegarder le fichier avec la nouvelle colonne "radical"
output_file_path = os.path.join(output_directory, 'Infinitifs.tsv')
merged_df.to_csv(output_file_path, sep='\t', index=False)

print("Fichier créé")

Fichier créé


##  <span style="color: #FFFFE0; font-weight: bold;">Epuration des non-correspondances</span>

In [6]:
# Cette étape consiste à ne garder que les radicaux des formes non-correspondantes pour chaque auteur

import os

# Chemins des répertoires
terminations_dir_path = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Localiser locuteurs\Terminaisons"
words_dir_path_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Poitevin\Formes trouvées\Non-correspondances"
output_dir_path_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Poitevin\Epuration"
words_dir_path_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Textes locuteurs\Saintongeais\Formes trouvées\Non-correspondances"
output_dir_path_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Saintongeais\Epuration"

# Fonction pour lire les terminaisons infinitives
def extract_terminations(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        terminations = []
        for line in file:
            if line.startswith("* 1inf :"):
                termination = line.split(":")[1].strip()
                terminations.append(termination)
        return terminations

# Fonction pour traiter les mots
def process_words(file_path, terminations):
    with open(file_path, 'r', encoding='utf-8') as file:
        processed_words = []
        for word in file:
            word = word.strip()
            for termination in terminations:
                if word.endswith(termination):
                    processed_word = word[:-len(termination)]
                    processed_words.append(processed_word)
                    break
            else:
                processed_words.append(word)
        return processed_words

# Fonction pour filtrer les mots de 3 caractères ou moins
def filter_short_words(words):
    return [word for word in words if len(word) > 3]

# Fonction pour enregistrer les résultats dans un fichier TSV sans doublons et triés
def save_to_tsv(file_path, processed_words, terminations):
    unique_words = list(set(processed_words))  # Utiliser un ensemble pour éliminer les doublons
    unique_words.sort()  # Trier les mots dans l'ordre alphabétique
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write("Term: " + ", ".join(terminations) + "\n")
        file.write("\n".join(unique_words))

# Fonction pour traiter les fichiers dans un répertoire donné
def process_directory(words_dir_path, output_dir_path):
    for filename in os.listdir(words_dir_path):
        if filename.endswith("_X.txt"):
            terminations_filename = filename.replace("_X", "")
            terminations_file_path = os.path.join(terminations_dir_path, terminations_filename)

            if not os.path.exists(terminations_file_path):
                print(f"Le fichier de terminaisons {terminations_filename} n'existe pas.")
                continue

            words_file_path = os.path.join(words_dir_path, filename)
            output_tsv_path = os.path.join(output_dir_path, filename.replace("_X.txt", ".tsv"))

            terminations = extract_terminations(terminations_file_path)
            processed_words = process_words(words_file_path, terminations)
            filtered_words = filter_short_words(processed_words)
            save_to_tsv(output_tsv_path, filtered_words, terminations)

# Traiter les fichiers Poitevin et Saintongeais
process_directory(words_dir_path_poitevin, output_dir_path_poitevin)
process_directory(words_dir_path_saintongeais, output_dir_path_saintongeais)

print("Traitement terminé pour tous les fichiers Poitevin et Saintongeais.")

Traitement terminé pour tous les fichiers Poitevin et Saintongeais.


##  <span style="color: #FFFFE0; font-weight: bold;">Correspondances avec le français</span>

In [7]:
import csv
import os

# Chemins des répertoires pour Poitevin
input_dir_path_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Poitevin\Epuration"
output_dir_path_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Poitevin\Correspondances"
non_correspondances_dir_path_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Poitevin\Non-correspondances"

# Chemins des répertoires pour Saintongeais
input_dir_path_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Saintongeais\Epuration"
output_dir_path_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Saintongeais\Correspondances"
non_correspondances_dir_path_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Saintongeais\Non-correspondances"

# Chemin du fichier des radicaux et lemmes
radicaux_file_path = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Lexiques\Lexiques radicaux infinitifs\Infinitifs.tsv"

# Lire les mots du fichier TSV en ignorant la ligne "Term:"
def lire_mots(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [line.strip() for line in file if not line.startswith("Term:")]

# Lire les terminaisons verbales du fichier
def lire_terminaisons(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            if line.startswith("Term:"):
                return line.split(":")[1].strip().replace(',', ' ').split()
    return []

# Lire les radicaux et les lemmes du fichier TSV
def lire_radicaux_et_lemmes(file_path):
    radicaux_lemmes = {}
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file, delimiter='\t')
        for row in reader:
            radicaux_lemmes[row['radical']] = row['lemme']
    return radicaux_lemmes

# Trouver les correspondances et les non-correspondances
def trouver_correspondances(mots, radicaux_lemmes):
    correspondances = []
    non_correspondances = []
    for mot in mots:
        if mot in radicaux_lemmes:
            lemme = radicaux_lemmes[mot]
            correspondances.append((mot, lemme))
        else:
            non_correspondances.append(mot)
    return correspondances, non_correspondances

# Appliquer les terminaisons aux mots
def appliquer_terminaisons(mots, terminaisons):
    mots_avec_terminaisons = set()
    for mot in mots:
        for terminaison in terminaisons:
            mot_avec_terminaison = mot + terminaison
            mots_avec_terminaisons.add(mot_avec_terminaison)
    return sorted(mots_avec_terminaisons)

# Appliquer les terminaisons aux correspondances
def appliquer_terminaisons_aux_correspondances(correspondances, terminaisons):
    correspondances_avec_terminaisons = set()
    for mot_ps, mot_fr in correspondances:
        for terminaison in terminaisons:
            mot_avec_terminaison = mot_ps + terminaison
            correspondances_avec_terminaisons.add((mot_avec_terminaison, mot_fr))
    return sorted(correspondances_avec_terminaisons)

# Enregistrer les correspondances dans un fichier TSV
def enregistrer_correspondances(file_path, correspondances):
    with open(file_path, 'w', encoding='utf-8', newline='') as file:
        writer = csv.writer(file, delimiter='\t')
        writer.writerow(['lemme', 'glose'])
        writer.writerows(correspondances)

# Enregistrer les non-correspondances avec terminaisons dans un fichier TSV
def enregistrer_non_correspondances(file_path, non_correspondances):
    with open(file_path, 'w', encoding='utf-8', newline='') as file:
        writer = csv.writer(file, delimiter='\t')
        for mot in non_correspondances:
            writer.writerow([mot])

# Traiter les fichiers dans le répertoire donné
def process_directory(input_dir_path, output_dir_path, non_correspondances_dir_path, radicaux_lemmes):
    for filename in os.listdir(input_dir_path):
        if filename.endswith(".tsv"):
            mots_file_path = os.path.join(input_dir_path, filename)
            mots = lire_mots(mots_file_path)
            terminaisons = lire_terminaisons(mots_file_path)
            correspondances, non_correspondances = trouver_correspondances(mots, radicaux_lemmes)

            # Appliquer les terminaisons aux correspondances
            correspondances_avec_terminaisons = appliquer_terminaisons_aux_correspondances(correspondances, terminaisons)

            # Chemin du fichier de sortie pour les correspondances
            output_file_path = os.path.join(output_dir_path, f"{filename}")
            enregistrer_correspondances(output_file_path, correspondances_avec_terminaisons)

            # Appliquer les terminaisons aux non-correspondances
            non_correspondances_avec_terminaisons = appliquer_terminaisons(non_correspondances, terminaisons)

            # Chemin du fichier de sortie pour les non-correspondances
            non_correspondances_file_path = os.path.join(non_correspondances_dir_path, f"{filename}")
            enregistrer_non_correspondances(non_correspondances_file_path, non_correspondances_avec_terminaisons)

# Exécuter les fonctions
radicaux_lemmes = lire_radicaux_et_lemmes(radicaux_file_path)
process_directory(input_dir_path_poitevin, output_dir_path_poitevin, non_correspondances_dir_path_poitevin, radicaux_lemmes)
process_directory(input_dir_path_saintongeais, output_dir_path_saintongeais, non_correspondances_dir_path_saintongeais, radicaux_lemmes)

print("Traitement terminé pour tous les fichiers Poitevin et Saintongeais.")

Traitement terminé pour tous les fichiers Poitevin et Saintongeais.


## <span style="color: #FFFFE0; font-weight: bold;">Comptage du nombre de nouveaux verbes à fléchir</span>

In [8]:
import os

def compter_verbes_tsv(dossier):
    total_verbes = 0
    for fichier in os.listdir(dossier):
        if fichier.endswith('.tsv'):
            chemin_fichier = os.path.join(dossier, fichier)
            with open(chemin_fichier, 'r', encoding='utf-8') as file:
                lignes = file.readlines()
                # Ne pas compter la première ligne (en-tête) et la dernière si elle est vide
                lignes_a_compter = lignes[1:]
                if lignes_a_compter and lignes_a_compter[-1].strip() == '':
                    lignes_a_compter = lignes_a_compter[:-1]
                total_verbes += len(lignes_a_compter)
    return total_verbes

# Chemins des dossiers
emprunts_saintongeais = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Saintongeais\Correspondances"
emprunts_poitevin = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Poitevin\Correspondances"

# Compter les verbes
total_saintongeais = compter_verbes_tsv(emprunts_saintongeais)
total_poitevin = compter_verbes_tsv(emprunts_poitevin)

total_ensemble = total_saintongeais + total_poitevin

# Afficher les résultats
print(f"Infinitifs en poitevin: {total_poitevin}")
print(f"Infinitifs en saintongeais: {total_saintongeais}")
print(f"Total d'infinitifs: {total_ensemble}")

Infinitifs en poitevin: 11956
Infinitifs en saintongeais: 4388
Total d'infinitifs: 16344


## <span style="color: #FFFFE0; font-weight: bold;">Comptage du nombre de formes à générer</span>

In [11]:
import re
import os

def count_terminations(termination_file_path):
    """Compte le nombre de terminaisons dans le fichier donné, en ignorant 'X'."""
    with open(termination_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        # Trouver toutes les lignes qui commencent par "* 1" et extraire les terminaisons
        termination_lines = re.findall(r'\* 1\w* : ([^\n]*)', content)
        # Compter toutes les terminaisons, en ignorant "X"
        terminations_count = 0
        for line in termination_lines:
            terms = [term.strip() for term in line.split(',') if term.strip() != "X"]
            terminations_count += len(terms)
        return terminations_count

def count_valid_lines_tsv(correspondance_file_path):
    """Compte le nombre de lignes valides dans le fichier de correspondance TSV, en ignorant la première ligne et la dernière si elle est vide."""
    with open(correspondance_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    # Ignorer la première ligne (en-tête) et la dernière ligne si elle est vide
    valid_lines = [line for line in lines[1:] if line.strip()]
    return len(valid_lines)

def calculate_conjugations(terminations_dir, correspondance_dir_tsv):
    """Calcule le nombre de conjugaisons possibles pour chaque fichier de terminaison."""
    conjugation_counts = {}
    # Obtenir une liste de tous les fichiers de terminaison
    termination_files = [f for f in os.listdir(terminations_dir) if f.endswith('.txt')]
    for termination_file in termination_files:
        base_file_name = os.path.splitext(termination_file)[0]
        # Trouver les fichiers de correspondance TSV correspondants
        corresponding_files_tsv = [f for f in os.listdir(correspondance_dir_tsv) if f.startswith(base_file_name) and f.endswith('.tsv')]
        termination_file_path = os.path.join(terminations_dir, termination_file)
        # Compter les terminaisons
        num_terminations = count_terminations(termination_file_path)
        # Calculer les conjugaisons pour les fichiers TSV
        for correspondance_file in corresponding_files_tsv:
            correspondance_file_path = os.path.join(correspondance_dir_tsv, correspondance_file)
            num_valid_lines = count_valid_lines_tsv(correspondance_file_path)
            conjugations = num_terminations * num_valid_lines
            conjugation_counts[correspondance_file] = conjugations
    return conjugation_counts

# Dossiers
terminations_dir = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Localiser locuteurs\Terminaisons"
correspondance_dir_poitevin_tsv = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Poitevin\Correspondances"
correspondance_dir_saintongeais_tsv = r"C:\Users\louis\OneDrive - Université de Poitiers\Fac\Master\Mémoire\Variation flexions\Emprunts\Saintongeais\Correspondances"

# Calculer les conjugaisons pour le Poitevin
conjugation_counts_poitevin = calculate_conjugations(terminations_dir, correspondance_dir_poitevin_tsv)

# Calculer les conjugaisons pour le Saintongeais
conjugation_counts_saintongeais = calculate_conjugations(terminations_dir, correspondance_dir_saintongeais_tsv)

# Afficher le résultat total pour le Poitevin
total_conjugations_poitevin = sum(conjugation_counts_poitevin.values())
print(f"Nombre total de conjugaisons possibles (Poitevin) : {total_conjugations_poitevin}")

# Afficher le résultat total pour le Saintongeais
total_conjugations_saintongeais = sum(conjugation_counts_saintongeais.values())
print(f"Nombre total de conjugaisons possibles (Saintongeais) : {total_conjugations_saintongeais}")

# Calculer et afficher le total général
total_conjugations_overall = total_conjugations_poitevin + total_conjugations_saintongeais
print(f"\nNombre total de conjugaisons possibles (Poitevin + Saintongeais) : {total_conjugations_overall}")

Nombre total de conjugaisons possibles (Poitevin) : 497580
Nombre total de conjugaisons possibles (Saintongeais) : 178099

Nombre total de conjugaisons possibles (Poitevin + Saintongeais) : 675679
